# AlphaGenome scoring for Benchmate (run in Google Colab)

Scores the variant-framed ERAD hypotheses with **AlphaGenome**, then exports a
`alphagenome_scores.json` file you bring back to Benchmate.

Why Colab: AlphaGenome needs Python 3.10+, which Colab already has — no local
install headaches. Same pattern as your Geneformer notebook.

**Steps:** run each cell top to bottom. You'll need a free AlphaGenome API key
from https://www.alphagenomedocs.com/ .


### 1. Install AlphaGenome


In [ ]:
!pip install -q alphagenome


### 2. Paste your AlphaGenome API key
(free, non-commercial — from alphagenomedocs.com)


In [ ]:
import getpass
API_KEY = getpass.getpass('AlphaGenome API key: ')


### 3. The variants to score
These mirror `benchmark/gold_set_variants.py`. **Coordinates are placeholders** —
replace each with a real regulatory variant before trusting the numbers.


In [ ]:
VARIANTS = [
    dict(label='SEL1L_prom',       chrom='chr14', pos=81_000_000, ref='C', alt='T'),
    dict(label='HERPUD1_enh',      chrom='chr16', pos=56_900_000, ref='G', alt='A'),
    dict(label='EDEM1_5utr',       chrom='chr3',  pos=5_230_000,  ref='A', alt='G'),
    dict(label='DERL1_intron',     chrom='chr8',  pos=123_900_000, ref='T', alt='C'),
    dict(label='SYVN1_intergenic', chrom='chr11', pos=64_900_000, ref='G', alt='C'),
    dict(label='XBP1_desert',      chrom='chr22', pos=28_790_000, ref='A', alt='T'),
]


### 4. Score each variant
For each variant we predict RNA-seq tracks for the reference vs alternate allele
and take the mean absolute change as the effect magnitude (larger = bigger
predicted regulatory effect).

> If a call errors, the AlphaGenome SDK arg names may have changed — check the
> Quick Start at alphagenomedocs.com and adjust this cell.


In [ ]:
import numpy as np
from alphagenome.data import genome
from alphagenome.models import dna_client

model = dna_client.create(API_KEY)
WINDOW = 131_072            # bp window around each variant
OUT = dna_client.OutputType.RNA_SEQ

scores = {}
for v in VARIANTS:
    variant = genome.Variant(chromosome=v['chrom'], position=v['pos'],
                             reference_bases=v['ref'], alternate_bases=v['alt'])
    interval = variant.reference_interval.resize(WINDOW)
    pred = model.predict_variant(interval=interval, variant=variant,
                                 requested_outputs=[OUT], ontology_terms=[])
    ref = np.asarray(pred.reference.get(OUT).values)
    alt = np.asarray(pred.alternate.get(OUT).values)
    scores[v['label']] = float(np.mean(np.abs(alt - ref)))
    print(f"{v['label']:18} -> {scores[v['label']]:.4f}")


### 5. Export the scores
Downloads `alphagenome_scores.json`. Put it in your Benchmate `benchmark/` folder,
then run `python -m benchmark.build_variant_scores` locally to add the Elo column
and produce `variant_scores.json` for the correlation (Benchmark tab, section 7).


In [ ]:
import json
from google.colab import files
with open('alphagenome_scores.json', 'w') as f:
    json.dump(scores, f, indent=2)
print(json.dumps(scores, indent=2))
files.download('alphagenome_scores.json')
